# LLM az-ornekli - Turkce Metin Siniflandirma Kiyaslamasi

**Katman:** Uretici taban

Buyuk dil modeli few-shot siniflandirma - DepremKit'in GPT-3.5 ile etiketledigi gibi; artik beklenen bir kiyaslama. Claude / OpenAI / yerel model uzerinden calisir.

Gorev: Turkce deprem ihbar metni -> 5 ust kategori (+ opsiyonel 1-5 aciliyet).  
Veri: `data/` (bkz `data/README.md`).  
Butun defterler `common.py` uzerinden AYNI bolme ve AYNI metrikleri kullanir.  
Metrik: Dogruluk, makro-F1, sinif bazli P/R, konfuzyon matrisi, **yanlis negatif orani**.

In [ ]:
# === COLAB KURULUM ===  (yerelde calistirinca atlanir)
import os, sys
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB and not os.path.exists('/content/QuakeMind'):
    !git clone --depth 1 https://github.com/UtBird/QuakeMind.git /content/QuakeMind
if IN_COLAB:
    %cd /content/QuakeMind/model_benchmarks/04_turkce_metin
    %pip install -q anthropic
print('cwd:', os.getcwd())

## 1. Veri

In [ ]:
import sys, warnings; warnings.filterwarnings('ignore')
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))
import numpy as np, pandas as pd
import common as C

train_df, val_df, test_df = C.load_text_dataset()
labels, l2i, i2l = C.label_maps(train_df)
print('siniflar:', labels)
print('train/val/test:', len(train_df), len(val_df), len(test_df))
y_test = test_df['label'].map(l2i).values

## 2. Prompt + istemci

In [ ]:
MODEL_NAME = "llm_few_shot"
LLM_PROVIDER = 'anthropic'      # 'anthropic' | 'openai'
LLM_MODEL = 'claude-sonnet-5'   # OpenAI icin ör. 'gpt-4o-mini'
N_SHOT = 6
PARAMS = dict(provider=LLM_PROVIDER, model=LLM_MODEL, n_shot=N_SHOT)

shots = train_df.groupby('label', group_keys=False).apply(lambda g: g.head(max(1, N_SHOT//len(labels))))
shot_txt = '\n'.join(f"Metin: {r.text}\nKategori: {r.label}" for r in shots.itertuples())
SYS = ('Turkce deprem ihbar metnini su kategorilerden birine ata: '
       + ', '.join(labels) + '. Sadece kategori adini yaz.')

def classify(text):
    prompt = f"{shot_txt}\n\nMetin: {text}\nKategori:"
    if LLM_PROVIDER == 'anthropic':
        import anthropic
        c = anthropic.Anthropic()
        r = c.messages.create(model=LLM_MODEL, max_tokens=10, system=SYS,
                              messages=[{'role': 'user', 'content': prompt}])
        out = r.content[0].text.strip().lower()
    else:
        from openai import OpenAI
        r = OpenAI().chat.completions.create(model=LLM_MODEL, max_tokens=10,
            messages=[{'role': 'system', 'content': SYS}, {'role': 'user', 'content': prompt}])
        out = r.choices[0].message.content.strip().lower()
    for l in labels:
        if l in out: return l
    return labels[-1]

## 3. Test kumesini siniflandir (API anahtari gerekli)

In [ ]:
from tqdm import tqdm
preds = [classify(t) for t in tqdm(test_df['text'].tolist())]
y_pred = np.array([l2i[p] for p in preds])

## 4. Sonuc

In [ ]:
m = C.classification_metrics(y_test, y_pred, labels)
print('accuracy :', round(m['accuracy'], 4))
print('macro-F1 :', round(m['macro_f1'], 4))
print('macro yanlis-negatif orani:', round(m['macro_fnr'], 4))
print()
print(m['report'])
lb = C.save_result(MODEL_NAME, m, extra={'params': PARAMS})
lb

In [ ]:
import matplotlib.pyplot as plt
cm = np.array(m['confusion_matrix'])
fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(cm, cmap='Blues')
ax.set_xticks(range(len(labels))); ax.set_xticklabels(labels, rotation=45, ha='right')
ax.set_yticks(range(len(labels))); ax.set_yticklabels(labels)
for (i, j), v in np.ndenumerate(cm): ax.text(j, i, int(v), ha='center', va='center')
ax.set_xlabel('tahmin'); ax.set_ylabel('gercek'); ax.set_title(f'{MODEL_NAME} konfuzyon')
plt.tight_layout(); plt.savefig(f'results/{MODEL_NAME}_cm.png', dpi=140); plt.show()